In [ ]:
import pandas as pd
import yfinance as yf
import sys

print("Python 버전:", sys.version)
print("실행 경로:", sys.executable)
print("Quant Regime Lab 시작!")
print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)


In [ ]:
# 1. spy의 과거 주식 데이터 다운로드
df = yf.download(
    "SPY",
    start="2010-01-01",
    end="2026-11-16",
    auto_adjust=False,
    multi_level_index=False,
    progress=False    
)

# 2.  데이터가 정상적으로 수집됐는지 확인
if df.empty:
    raise ValueError("데이터 다운로드에 실패했습니다.")

# 3. 데이터 기본 정보 확인
print("총 거래일 수:", len(df))
print("데이터 시작일:", df.index.min())
print("데이터 종료일:", df.index.max())

# 4. 최근 5거래일 데이터 확인
display(df.tail())

#5. 열별 결측치 확인
print("결측치 개수:")
print(df.isna().sum())

In [ ]:
import numpy as np
# 1. 일별 단순 수익률 계산
df["daily_return"] = df["Adj Close"].pct_change() # 오늘 주가가 전 거래일보다 얼마나 상승하거나 하락했는지 게산 

# 2. 일별 로그수익률 계산
df["log_return"] = np.log( #np.log() -> 자연로그 계산 

    # / 오늘 가격을 어제 가격으로 나누기 
    df["Adj Close"] / df["Adj Close"].shift(1) #shift(1) : 데이터를 한 행 이동 , 현재 행에 전 거래일 가격이 오도록 만듦
)

# 3. 최근 20거래일의 변동성 계산 #rolling(20) : 20개씩 묶어서 계산 , 일정한 크기의 구간을하나씩 이동하면서 계산하는 함수 
df["volatility_20"] = df["daily_return"].rolling(20).std() #std() : 표준편차 계산 

# 4. 변동성을 연간 기준으로 환산  #np.sprt(252) -> 252의 제곱근 
df["annual_volatility"] = df["volatility_20"] * np.sqrt(252)

#5. 계산 결과 확인
display(df[[
    "Adj Close", #수정종가
    "daily_return", #일별 단순수익률
    "log_return", #일별 로그수익률
    "volatility_20", # 최근 20거래일 수익률의 표준편차
    "annual_volatility"  #연간 기준으로 환산한 변동성 
]].tail())

In [ ]:
#step 3 : spy 주가 및 변동성 시각화 
# 주가 변동성 그래프 생성

import matplotlib.pyplot as plt #그래프를 그리는데 활용됨 

#1. 그래프 2개 생성 
fig , ax = plt.subplots(  #그래프를 그릴 공간을 만들어줌 subplots 하나의 figure 안에 여러 그래프를 배치할 수 있도록 도와줌 
    2, 1,                 #fig 와 ax --> fig : 전체 그림인 figure(도화지) , ax : 실제 그래프를 그리는 공간인 axes(도화지 안에 그래프를 그릴 구역)
    figsize=(14,9), # 가로 14인치 세로 9인치
    sharex=True  # 두 그래프의 x축을 공유하겠다. 
)

# 2. spy 수정종가 그래프 
ax[0].plot(df.index, df["Adj Close"])
ax[0].set_title("SPY Adjusted Close Price")
ax[0].set_ylabel("Price (USD)")
ax[0].grid(True, alpha=0.3)

# 3. spy 연율화 변동성 그래프
ax[1].plot(
    df.index,
    df["annual_volatility"] * 100
)
ax[1].set_title("SPY 20-Day Annualized Volatility")
ax[1].set_ylabel("Volatility (%)")
ax[1].set_xlabel("date")
ax[1].grid(True, alpha=0.3)

# 4. 그래프 출력
plt.tight_layout()
plt.show()